# Analyst Agent

The **Analyst Agent** is the final convergence node — it runs after all parallel branches complete.

Uses **gpt-4o** (the flagship model) to synthesize:
- `news_summary` from the News Agent
- `financials` from the Financials Agent  
- `sentiment_score` from the Sentiment Agent

**Output:** A structured investment report with:
- `RECOMMENDATION` — BUY / HOLD / SELL
- `PRICE TARGET` — 12-month target with justification
- `TOP 3 REASONS` — data-backed thesis points
- `KEY RISKS` — what could go wrong
- `SUMMARY` — 2-3 sentence executive summary

**Output state key:** `report` (formatted string)

In [ ]:
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()

# gpt-4o — only used here, for final synthesis
_llm = ChatOpenAI(model="gpt-4o", temperature=0.1)


def _sentiment_label(score: float) -> str:
    if score is None:
        return "Unknown"
    if score < 0.2:
        return "Very Bearish"
    elif score < 0.4:
        return "Bearish"
    elif score < 0.6:
        return "Neutral"
    elif score < 0.8:
        return "Bullish"
    return "Very Bullish"


def analyst_node(state: dict) -> dict:
    """
    Synthesize news, financials, and sentiment into a structured analyst report.
    Uses gpt-4o — the only agent that uses the flagship model.
    Returns: report (formatted markdown string).
    """
    ticker = state["ticker"]
    news_summary = state.get("news_summary") or "No news available."
    financials = state.get("financials") or {}
    sentiment_score = state.get("sentiment_score") or 0.5

    print(f"[Analyst] Synthesizing final report for {ticker}...")

    sentiment_label = _sentiment_label(sentiment_score)
    fin_str = json.dumps(financials, indent=2, default=str)

    prompt = f"""You are a senior equity research analyst at a top-tier investment bank.
Generate a structured investment report based on the data below.

TICKER: {ticker}

NEWS SUMMARY:
{news_summary}

FINANCIAL DATA:
{fin_str}

SENTIMENT SCORE: {sentiment_score:.2f} / 1.00 ({sentiment_label})

Generate the report with EXACTLY these sections and headers:

## RECOMMENDATION
BUY / HOLD / SELL  (pick one and explain in one sentence)

## PRICE TARGET
12-month price target with a brief valuation justification (1-2 sentences)

## TOP 3 REASONS
1. [Reason with specific data point]
2. [Reason with specific data point]
3. [Reason with specific data point]

## KEY RISKS
1. [Risk 1]
2. [Risk 2]
3. [Risk 3]

## SUMMARY
[2-3 sentence executive summary for a busy portfolio manager]

Be specific, cite numbers from the financial data, and maintain a professional analyst tone."""

    response = _llm.invoke([HumanMessage(content=prompt)])
    report = response.content.strip()

    print(f"[Analyst] Report generated ({len(report)} chars)")
    return {"report": report}


In [ ]:
if __name__ == "__main__":
    # --- Demo: run analyst with sample data ---
    sample_state = {
        "ticker": "NVDA",
        "news_summary": (
            "• Nvidia reported Q4 revenue of $22.1B, up 265% YoY, driven by data center demand\n"
            "• Blackwell GPU architecture shipping at scale; backlog extends into 2026\n"
            "• Jensen Huang confirms sovereign AI deals with 40+ governments"
        ),
        "financials": {
            "ticker": "NVDA",
            "company_name": "NVIDIA Corporation",
            "current_price": 875.0,
            "pe_ratio": 72.4,
            "forward_pe": 35.2,
            "revenue_growth": 2.22,
            "market_cap": 2_150_000_000_000,
            "52_week_high": 974.0,
            "52_week_low": 461.0,
            "beta": 1.65,
            "analyst_target_price": 1050.0,
        },
        "sentiment_score": 0.85,
    }
    result = analyst_node(sample_state)
    print(result["report"])
